# Qwen3-VL-30B-A3B MoE Expert Activation Analysis on LEGOLite

Designed for Google Colab with NVIDIA G4 Blackwell GPU (96GB VRAM).

Runs the model locally on the LEGOLite benchmark (height, position, rotation, ordering) and captures which of the 128 MoE experts activate for each question. Exports per-question expert frequency maps as JSON grouped by spatial category.

**Model architecture (from config.json):**
- 48 decoder layers, all with MoE blocks (`decoder_sparse_step=1`)
- 128 experts per layer, 8 active per token (`num_experts_per_tok=8`)
- Router: `Qwen3VLMoeTextTopKRouter` accessed via `layer.mlp.gate`

In [ ]:
!pip install -q transformers>=4.51.0 accelerate torch pillow pandas tqdm qwen-vl-utils

In [ ]:
import os
import json
import string
import base64
from io import BytesIO
from collections import Counter, defaultdict

import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"bfloat16 support: {torch.cuda.is_bf16_supported()}")

In [ ]:
MODEL_ID = "Qwen/Qwen3-VL-30B-A3B-Instruct"

from transformers import Qwen3VLMoeForConditionalGeneration, AutoProcessor

print(f"Loading {MODEL_ID}...")
model = Qwen3VLMoeForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map="auto",
)
model.eval()

processor = AutoProcessor.from_pretrained(MODEL_ID)

text_config = model.config.text_config
print(f"\nModel loaded successfully.")
print(f"  Layers: {text_config.num_hidden_layers}")
print(f"  Experts: {text_config.num_experts}")
print(f"  Top-K: {text_config.num_experts_per_tok}")
print(f"  Hidden size: {text_config.hidden_size}")
print(f"  Decoder sparse step: {text_config.decoder_sparse_step}")

In [ ]:
NUM_LAYERS = text_config.num_hidden_layers    # 48
NUM_EXPERTS = text_config.num_experts          # 128
TOP_K = text_config.num_experts_per_tok        # 8

expert_log = defaultdict(lambda: Counter())
current_question_id = None
hooks = []


def make_router_hook(layer_idx):
    """
    Forward hook for Qwen3VLMoeTextTopKRouter.
    Router forward returns (router_logits, router_scores, router_indices).
    router_indices shape: (seq_len, top_k) with the selected expert IDs.
    """
    def hook_fn(module, input, output):
        _, _, router_indices = output
        indices = router_indices.detach().cpu().flatten().tolist()
        key = (current_question_id, layer_idx)
        expert_log[key].update(indices)
    return hook_fn


def find_decoder_layers():
    """Dynamically discover the decoder layer list in the model."""
    # Try common paths for Qwen3 VL MoE
    for attr_path in ["model.layers", "model.model.layers"]:
        candidate = model
        try:
            for attr in attr_path.split("."):
                candidate = getattr(candidate, attr)
            if hasattr(candidate, '__len__') and len(candidate) == NUM_LAYERS:
                print(f"Found decoder layers at: model.{attr_path} ({len(candidate)} layers)")
                return candidate
        except AttributeError:
            continue

    # Fallback: search all named modules for the layer list
    for name, module in model.named_modules():
        if hasattr(module, '__len__'):
            try:
                if len(module) == NUM_LAYERS and hasattr(module[0], 'mlp'):
                    print(f"Found decoder layers at: {name} ({len(module)} layers)")
                    return module
            except (TypeError, IndexError):
                continue

    raise RuntimeError("Could not find decoder layers. Print model structure with: print(model)")


decoder_layers = find_decoder_layers()

# Verify MoE gates exist
moe_layer_count = sum(1 for layer in decoder_layers if hasattr(layer.mlp, 'gate'))
print(f"MoE layers (with .mlp.gate): {moe_layer_count} / {NUM_LAYERS}")


def register_hooks():
    """Attach hooks to every MoE router gate."""
    global hooks
    remove_hooks()
    for layer_idx, layer in enumerate(decoder_layers):
        if hasattr(layer.mlp, 'gate'):
            hook = layer.mlp.gate.register_forward_hook(make_router_hook(layer_idx))
            hooks.append(hook)
    print(f"Registered {len(hooks)} router hooks.")


def remove_hooks():
    global hooks
    for hook in hooks:
        hook.remove()
    hooks = []

In [ ]:
LEGO_TSV_URL = "https://opencompass.openxlab.space/utils/VLMEval/LEGO.tsv"
LITE_CATEGORIES = ["height", "position", "rotation", "ordering"]
DATA_DIR = "/content/lego_data"
IMG_DIR = "/content/lego_images"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

tsv_path = os.path.join(DATA_DIR, "LEGO.tsv")
if not os.path.exists(tsv_path):
    print("Downloading LEGO.tsv (~142MB)...")
    import urllib.request
    urllib.request.urlretrieve(LEGO_TSV_URL, tsv_path)
    print("Download complete.")
else:
    print("LEGO.tsv already downloaded.")

df_full = pd.read_csv(tsv_path, sep="\t")
df_lite = df_full[df_full["category"].isin(LITE_CATEGORIES)].reset_index(drop=True)
print(f"\nLEGOLite: {len(df_lite)} questions")
print(df_lite["category"].value_counts().to_string())

# Pre-decode and save all images to disk
# The 'image' column can be: (a) base64 string, or (b) an index referencing another row
print(f"\nDecoding images to {IMG_DIR}...")
decoded_count = 0
skipped_count = 0

# Build a lookup: index -> base64 data (from full dataset, for cross-references)
full_index_map = {str(row["index"]): row for _, row in df_full.iterrows()}

def save_image_for_row(row):
    """Decode and save image(s) for a row. Returns list of saved file paths."""
    idx = str(row["index"])
    img_col = row.get("image", None)
    paths = []

    if pd.isna(img_col) or img_col is None:
        return paths

    img_str = str(img_col)

    # Check if it's a cross-reference (short string that's just a number)
    if len(img_str) < 20:
        try:
            ref_idx = str(int(float(img_str)))
            if ref_idx in full_index_map:
                ref_row = full_index_map[ref_idx]
                img_str = str(ref_row["image"])
            else:
                return paths
        except (ValueError, TypeError):
            return paths

    # Try to decode as base64
    out_path = os.path.join(IMG_DIR, f"{idx}.png")
    if os.path.exists(out_path):
        return [out_path]

    try:
        img_bytes = base64.b64decode(img_str)
        img = Image.open(BytesIO(img_bytes)).convert("RGB")
        img.save(out_path)
        paths.append(out_path)
    except Exception:
        pass

    return paths

image_path_map = {}  # question index -> list of image file paths
for row_idx in range(len(df_lite)):
    row = df_lite.iloc[row_idx]
    idx = str(row["index"])
    saved = save_image_for_row(row)
    if saved:
        decoded_count += 1
    else:
        skipped_count += 1
    image_path_map[idx] = saved

print(f"Decoded: {decoded_count}, Skipped (no image): {skipped_count}")

In [ ]:
def build_messages(row):
    """
    Build Qwen3-VL chat messages for a single LEGO question.
    Returns the messages list for processor.apply_chat_template.
    """
    idx = str(row["index"])
    question = row["question"]
    question_type = row["question_type"]

    option_cols = [option_letter for option_letter in string.ascii_uppercase if option_letter in row.index and pd.notna(row[option_letter])]

    prompt = f"Question: {question}\n"
    if option_cols:
        prompt += "Options:\n"
        for option_letter in option_cols:
            option_value = row[option_letter]
            if isinstance(option_value, str) and "<image" in option_value:
                prompt += f"{option_letter}. [see image]\n"
            else:
                prompt += f"{option_letter}. {option_value}\n"

    if question_type == "sort":
        prompt += "Please respond with only the sequence of letters (e.g., 'BDAC') that correctly orders the steps.\n"
    else:
        prompt += "Please respond with only the letter of the correct answer.\n"

    content = []
    img_paths = image_path_map.get(idx, [])
    for image_path in img_paths:
        content.append({"type": "image", "image": f"file://{image_path}"})
    content.append({"type": "text", "text": prompt})

    return [{"role": "user", "content": content}]


# Quick test: build messages for the first question
test_row = df_lite.iloc[0]
test_msgs = build_messages(test_row)
print(f"Test question [{test_row['category']}]: {test_row['question'][:80]}...")
print(f"Images: {len(image_path_map.get(str(test_row['index']), []))}")
print(f"Message content items: {len(test_msgs[0]['content'])}")

In [ ]:
# Determine device for inputs
if hasattr(model, 'device') and model.device.type != 'meta':
    input_device = model.device
else:
    input_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Input device: {input_device}")

register_hooks()

results = []
all_expert_data = {}

print(f"\nRunning inference on {len(df_lite)} questions...\n")

for row_idx in tqdm(range(len(df_lite)), desc="LEGOLite"):
    row = df_lite.iloc[row_idx]
    q_id = str(row["index"])
    category = row["category"]
    answer_gt = str(row["answer"])

    current_question_id = q_id

    try:
        messages = build_messages(row)

        # Qwen3-VL recommended inference: single-call apply_chat_template
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(input_device)

        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=64)

        # Trim input tokens from output (official Qwen3-VL pattern)
        generated_ids = [
            full_output[len(full_input):] for full_input, full_output in zip(inputs["input_ids"], output_ids)
        ]
        prediction = processor.batch_decode(
            generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

    except Exception as error:
        prediction = f"ERROR: {error}"
        if row_idx < 3:  # Print first few errors for debugging
            print(f"\n  Error on q_id={q_id}: {error}")

    # Aggregate expert activations for this question across all layers
    question_expert_freq = Counter()
    per_layer = {}
    for layer_idx in range(NUM_LAYERS):
        key = (q_id, layer_idx)
        if key in expert_log:
            question_expert_freq.update(expert_log[key])
            per_layer[str(layer_idx)] = dict(expert_log[key])

    is_correct = prediction.strip().upper().startswith(answer_gt.strip().upper())

    all_expert_data[q_id] = {
        "question_id": q_id,
        "category": category,
        "prediction": prediction,
        "ground_truth": answer_gt,
        "correct": is_correct,
        "expert_frequency": {str(expert_id): count for expert_id, count in question_expert_freq.items()},
        "per_layer_experts": per_layer,
    }

    results.append({
        "question_id": q_id,
        "category": category,
        "prediction": prediction,
        "ground_truth": answer_gt,
        "correct": is_correct,
    })

    # Free memory for this question
    for log_key in [log_key for log_key in expert_log if log_key[0] == q_id]:
        del expert_log[log_key]
    torch.cuda.empty_cache()

remove_hooks()
print("\nInference complete.")

In [ ]:
results_df = pd.DataFrame(results)
print("=== LEGOLite Results (Qwen3-VL-30B-A3B) ===\n")

overall_acc = results_df["correct"].mean()
print(f"Overall accuracy: {overall_acc:.1%}")
print()

for cat in LITE_CATEGORIES:
    cat_df = results_df[results_df["category"] == cat]
    if len(cat_df) > 0:
        acc = cat_df["correct"].mean()
        print(f"  {cat:12s}: {acc:.1%} ({int(cat_df['correct'].sum())}/{len(cat_df)})")

# Show error count
error_count = results_df["prediction"].str.startswith("ERROR:").sum()
if error_count > 0:
    print(f"\n  Errors: {error_count} questions failed during inference")

In [ ]:
category_expert_freq = defaultdict(Counter)
for q_id, question_data in all_expert_data.items():
    cat = question_data["category"]
    freq = {int(expert_id): count for expert_id, count in question_data["expert_frequency"].items()}
    category_expert_freq[cat].update(freq)

print("=== Top 15 Most Active Experts per Category ===\n")

category_summary = {}
for cat in LITE_CATEGORIES:
    freq = category_expert_freq[cat]
    if not freq:
        print(f"{cat}: no data\n")
        continue
    top_experts = freq.most_common(15)
    total = sum(freq.values())
    category_summary[cat] = {
        "top_15_experts": [{"expert_id": expert_id, "activation_count": count} for expert_id, count in top_experts],
        "total_activations": total,
        "unique_experts_used": len(freq),
    }
    print(f"{cat} ({total:,} total activations, {len(freq)} unique experts):")
    for expert_id, count in top_experts:
        pct = count / total * 100
        print(f"  Expert {expert_id:3d}: {count:>10,} activations ({pct:.2f}%)")
    print()

In [ ]:
analysis_report = {
    "model": MODEL_ID,
    "benchmark": "LEGOLite",
    "categories": LITE_CATEGORIES,
    "num_questions": len(df_lite),
    "accuracy": {
        "overall": float(overall_acc),
        **{cat: float(results_df[results_df["category"] == cat]["correct"].mean())
           for cat in LITE_CATEGORIES}
    },
    "category_expert_summary": category_summary,
    "per_question_results": all_expert_data,
}

output_path = "/content/lego_moe_expert_analysis.json"
with open(output_path, "w") as output_file:
    json.dump(analysis_report, output_file, indent=2)

size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"Results exported to {output_path}")
print(f"File size: {size_mb:.1f} MB")
print(f"\nTo download: use Colab file browser (left panel) or:")
print(f"  from google.colab import files; files.download('{output_path}')")